# 04 — Automated Data Quality Checks

**Purpose:** Validate the persisted Silver and Gold outputs after transformation.

This notebook is designed to run as the final Lakeflow Job task. A failed assertion raises an exception, causing the quality task—and therefore the Job—to fail instead of silently publishing questionable results.

The project keeps several full-table checks for clarity. At much larger production scale, expensive checks would typically be scoped to affected partitions/dates or scheduled less frequently.


In [ ]:
from pyspark.sql import functions as F

SILVER_TABLE = "ecommerce_lakehouse.silver.transactions_clean"

GOLD_FUNNEL_TABLE = "ecommerce_lakehouse.gold.daily_funnel_metrics"
GOLD_REVENUE_TABLE = "ecommerce_lakehouse.gold.daily_revenue_metrics"
GOLD_PRODUCT_TABLE = "ecommerce_lakehouse.gold.product_daily_performance"

silver_df = spark.table(SILVER_TABLE)
funnel_df = spark.table(GOLD_FUNNEL_TABLE)
revenue_df = spark.table(GOLD_REVENUE_TABLE)
product_df = spark.table(GOLD_PRODUCT_TABLE)


## 1. Freshness

Gold should be refreshed through the same latest event date currently available in Silver.


In [ ]:
silver_max_date = (
    silver_df
        .agg(F.max("event_date").alias("max_date"))
        .first()["max_date"]
)

gold_max_date = (
    funnel_df
        .agg(F.max("event_date").alias("max_date"))
        .first()["max_date"]
)

assert silver_max_date == gold_max_date, (
    "Gold is not up to date. "
    f"Silver max date = {silver_max_date}, "
    f"Gold max date = {gold_max_date}"
)

print("Freshness check passed.")


## 2. Daily funnel/event metrics

The table grain is one row per day. The checks enforce a non-null key, unique daily grain, and non-negative event metrics.

The `*_rate` fields are event-count ratios rather than strict cohort conversion probabilities, so they are checked for non-negativity but are **not** incorrectly constrained to `<= 1`.


In [ ]:
assert (
    funnel_df
        .filter(F.col("event_date").isNull())
        .count()
    == 0
), "daily_funnel_metrics contains NULL event_date"

assert (
    funnel_df
        .groupBy("event_date")
        .count()
        .filter(F.col("count") > 1)
        .count()
    == 0
), "daily_funnel_metrics contains duplicate dates"

assert (
    funnel_df
        .filter(
            (F.col("total_events") < 0)
            | (F.col("views") < 0)
            | (F.col("carts") < 0)
            | (F.col("cart_removals") < 0)
            | (F.col("purchases") < 0)
        )
        .count()
    == 0
), "daily_funnel_metrics contains negative metrics"

assert (
    funnel_df
        .filter(
            (F.col("view_to_cart_rate") < 0)
            | (F.col("cart_to_purchase_rate") < 0)
            | (F.col("view_to_purchase_rate") < 0)
        )
        .count()
    == 0
), "daily_funnel_metrics contains negative event ratios"

print("Funnel quality checks passed.")


## 3. Daily revenue metrics

The table grain is one row per day. Monetary metrics must be non-negative and the daily key must remain unique.


In [ ]:
assert (
    revenue_df
        .filter(F.col("event_date").isNull())
        .count()
    == 0
), "daily_revenue_metrics contains NULL event_date"

assert (
    revenue_df
        .groupBy("event_date")
        .count()
        .filter(F.col("count") > 1)
        .count()
    == 0
), "daily_revenue_metrics contains duplicate dates"

assert (
    revenue_df
        .filter(
            (F.col("purchase_events") < 0)
            | (F.col("purchased_item_revenue") < 0)
            | (F.col("avg_purchased_item_price") < 0)
            | (F.col("unique_buyers") < 0)
        )
        .count()
    == 0
), "daily_revenue_metrics contains invalid monetary metrics"

print("Revenue quality checks passed.")


## 4. Product daily performance

The table grain is one row per `event_date + product_id`. Both key columns must be present and the composite grain must stay unique.


In [ ]:
assert (
    product_df
        .filter(
            F.col("event_date").isNull()
            | F.col("product_id").isNull()
        )
        .count()
    == 0
), "product_daily_performance contains NULL keys"

assert (
    product_df
        .groupBy("event_date", "product_id")
        .count()
        .filter(F.col("count") > 1)
        .count()
    == 0
), "product_daily_performance contains duplicate grain"

assert (
    product_df
        .filter(
            (F.col("views") < 0)
            | (F.col("carts") < 0)
            | (F.col("purchases") < 0)
            | (F.col("purchased_item_revenue") < 0)
            | (F.col("unique_users") < 0)
        )
        .count()
    == 0
), "product_daily_performance contains invalid metrics"

print("Product quality checks passed.")


## 5. Silver-to-Gold reconciliation

For the latest available date, the number of Silver purchase events must equal the Gold purchase count. This checks that the aggregation did not silently lose or duplicate purchase events.


In [ ]:
latest_date = silver_max_date

silver_latest_purchases = (
    silver_df
        .filter(
            (F.col("event_date") == latest_date)
            & (F.col("event_type") == "purchase")
        )
        .count()
)

gold_latest_purchases = (
    funnel_df
        .filter(F.col("event_date") == latest_date)
        .select("purchases")
        .first()["purchases"]
)

assert silver_latest_purchases == gold_latest_purchases, (
    f"Purchase reconciliation failed for {latest_date}: "
    f"Silver={silver_latest_purchases}, "
    f"Gold={gold_latest_purchases}"
)

print(
    f"Purchase reconciliation passed for {latest_date}: "
    f"{silver_latest_purchases}"
)


In [ ]:
print("✅ All data quality checks passed.")
